# Integrating Generative AI with the Diagnostic Pipeline

**SENTINEL-CXR** — Uncertainty-Aware Chest Radiograph Triage
Deep Learning (MAIB AI 114) · Prof Anshul Gupta · S P Jain School of Global Management, Dubai

| Group member | Student ID |
|---|---|
| Krishna Mathur | AS25DXB018 |
| Atharva Soundankar | AS25DXB020 |
| Yash Petkar | AS25DXB021 |

---

**Syllabus mapping — Week 11: Integration of Generative AI with Deep Learning**

Learning outcome D. Generative components are wired into the discriminative
pipeline in three places, and each is evaluated on whether it actually helps:

1. **GAN** — synthetic minority-class samples augmenting the classifier.
2. **VAE** — the distributional gate that rejects non-radiographs.
3. **LLM** — drafting the report from structured model output.

The third carries the real risk. A language model asked to describe a radiograph
will produce a fluent report containing findings the vision model never detected,
written in a register indistinguishable from a true finding. This notebook builds
and *attacks* the defence.


In [ ]:
# ── Environment ───────────────────────────────────────────────────────
# Runs on Colab free tier (T4). Nothing here needs a paid runtime.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "torchxrayvision", "scikit-learn", "seaborn"],
        check=False,
    )

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 20260812
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}")

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.grid": True, "grid.alpha": 0.25,
})
INSTRUMENT, STAT = "#2E9CB8", "#D64541"

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────
# NIH ChestX-ray14: 112,120 frontal radiographs, 30,805 patients, 14 labels.
# Kaggle: https://www.kaggle.com/datasets/nih-chest-xrays/data
#
# In Colab, the fastest route is the Kaggle API:
#   from google.colab import files; files.upload()      # kaggle.json
#   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#   !kaggle datasets download -d nih-chest-xrays/data -p /content/nih --unzip

DATA_DIR = os.environ.get("NIH_DIR", "/content/nih")
META = os.path.join(DATA_DIR, "Data_Entry_2017.csv")

PATHOLOGIES = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Effusion",
               "Emphysema","Fibrosis","Hernia","Infiltration","Mass","Nodule",
               "Pleural_Thickening","Pneumonia","Pneumothorax"]

def load_metadata(path=META):
    """Load the label CSV and expand `Finding Labels` into 14 binary columns."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    for p in PATHOLOGIES:
        df[p] = df["Finding Labels"].str.contains(p, regex=False).astype(int)
    df["Patient Age"] = pd.to_numeric(df["Patient Age"], errors="coerce")
    # Ages above ~100 in this dataset are data-entry errors, not centenarians.
    df = df[(df["Patient Age"] > 0) & (df["Patient Age"] < 100)]
    return df

def patient_disjoint_split(df, fracs=(0.70, 0.10, 0.20), seed=SEED):
    """Split by Patient ID — NEVER by image.

    A patient contributes 3-4 follow-up studies. Splitting by image places the
    same patient's scans on both sides of the boundary, so the model can
    memorise the patient rather than the pathology. Every metric then reports a
    number that will not survive contact with a new hospital. This is the most
    common methodological error in published work on ChestX-ray14.
    """
    patients = df["Patient ID"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    a, b = int(fracs[0] * n), int((fracs[0] + fracs[1]) * n)
    sets = (set(patients[:a]), set(patients[a:b]), set(patients[b:]))
    train, cal, test = (df[df["Patient ID"].isin(s)].copy() for s in sets)
    assert not (set(train["Patient ID"]) & set(test["Patient ID"])), "patient leak"
    return train, cal, test

## 1. Grounded generation

The model never sees the image. It transforms structured output into prose, nothing more.

In [ ]:
PATHOLOGIES_SET = set(PATHOLOGIES)

def build_prompt(findings, conformal, allowed):
    evidence = "\n".join(
        f"- {f['name'].replace('_',' ')}: probability {f['probability']:.3f}"
        for f in findings if f["included"]) or "- None above threshold"
    return f"""You are drafting FINDINGS and IMPRESSION for radiologist review.
You are NOT looking at an image. You have the structured output of a vision model.

DETECTED:
{evidence}

PREDICTION SET: {', '.join(conformal['prediction_set']) or 'empty'}
ABSTAINED: {conformal['abstained']}

RULES:
1. Mention ONLY these pathologies: {', '.join(allowed) or 'none'}.
2. Invent nothing — no findings, measurements, laterality, or history.
3. If abstained, say so and require radiologist review.
Write the report."""

print("Prompt template ready.")

## 2. The verifier

Prompt instructions are a request. Verification is a guarantee.

In [ ]:
import re

def verify_grounding(text, supported):
    """Reject text naming any pathology outside the supported set."""
    low = text.lower(); sup = {s.lower() for s in supported}
    for p in PATHOLOGIES:
        if p.lower() in sup: continue
        for variant in {p.lower(), p.replace("_", " ").lower()}:
            if re.search(rf"\b{re.escape(variant)}\b", low):
                return False, f"mentions unsupported finding: {p}"
    return True, ""

# Adversarial cases — these are the attacks that matter.
CASES = [
    ("Right pleural effusion is present.",          {"Effusion"},      True),
    ("A large pneumothorax is seen on the left.",   {"Effusion"},      False),
    ("No pneumothorax is seen.",                    {"Effusion"},      False),
    ("There is massive consolidation.",             {"Consolidation"}, True),
    ("CARDIOMEGALY IS PRESENT",                     {"Effusion"},      False),
    ("Marked pleural thickening noted.",            set(),             False),
]
print(f"{'verdict':>8}  {'expected':>8}  text")
for text, sup, expected in CASES:
    ok, _ = verify_grounding(text, sup)
    mark = "OK " if ok == expected else "FAIL"
    print(f"{str(ok):>8}  {str(expected):>8}  {mark}  {text[:44]}")

print("\nTwo cases deserve comment:")
print(" - 'massive consolidation' PASSES: 'mass' is a substring of 'massive',")
print("   but word-boundary matching correctly does not fire.")
print(" - 'No pneumothorax is seen' is REJECTED even though it is a negation.")
print("   A radiologist reading it infers the system looked for a pneumothorax")
print("   and ruled it out. If it was never assessed, that inference is false.")

## 3. Measuring hallucination rate

In [ ]:
def hallucination_experiment(n=100):
    """Rate of ungrounded findings, with and without the verifier.

    Run the generator over n studies and count how often it names a pathology
    outside the supported set. Reported in the ethics section of the report.

    The verifier makes the post-filter rate exactly zero *by construction* —
    it is a hard gate, not a probabilistic mitigation. The number worth
    reporting is the PRE-filter rate, because that is what a system without
    this defence would have shown a clinician.
    """
    print(hallucination_experiment.__doc__)

hallucination_experiment()

---

### References for this notebook

- Goodfellow, I. et al. (2014). Generative adversarial networks. *NeurIPS*.
- Ji, Z. et al. (2023). Survey of hallucination in natural language generation. *ACM Computing Surveys*.
- Singhal, K. et al. (2023). Large language models encode clinical knowledge. *Nature*.

---

*SENTINEL-CXR is a student research prototype. It is not a medical device and
must not be used for clinical decisions.*
